In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from typing import TypedDict,Literal
from langchain_core.messages import BaseMessage
from pathlib import Path

import os
load_dotenv()
llm_model = os.getenv("LLM_MODEL")

In [3]:
llm=ChatOpenAI(model=llm_model)
agent=create_agent(model=llm)

In [ ]:
agent_response=agent.invoke({
    'messages':[HumanMessage(content="Hi are properly working?")]
})

agent_response

{'messages': [HumanMessage(content='Hi are properly working?', additional_kwargs={}, response_metadata={}, id='90166d05-ea9f-463f-8f14-4a5de9cd8320'),
  AIMessage(content='Hi! Yes—I’m working properly.  \n\nHow can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 11, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Da4bmtiRyPbBamXHsRB3W0BTmQGEJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ddaa7-53cb-70a1-9dd0-99d5f7859e05-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 19, 'total_tokens': 30, 'input_token_details': {'audio': 0, '

In [5]:
class VaAnalysisState(TypedDict):
    messages:list[BaseMessage]
    issue_list:list[str]

class DetailedIssue(TypedDict):
    issue_category:Literal["VA","OTHER"]
    detailed_issue:str
    initial_fix:str
    bom_friendly_fix:str

In [6]:
print(Path.cwd())
jenkins_output_raw_text = Path("trivy_scan_input.txt").read_text(encoding="utf-8")
jenkins_output_raw_text


d:\DIALOG\DESKTOP\AI -Products\Project 07 - Automated vulnerability detection and remediation for CICD pipelines\PROPER_CODE_BASE\pipeline-security-autofix\dev_test\notebook


'===== build-image.log =====\n\n\n===== scan-dependencies.log =====\n\nReport Summary\n\n┌─────────┬──────┬─────────────────┐\n│ Target  │ Type │ Vulnerabilities │\n├─────────┼──────┼─────────────────┤\n│ pom.xml │ pom  │       30        │\n└─────────┴──────┴─────────────────┘\nLegend:\n- \'-\': Not scanned\n- \'0\': Clean (no security findings detected)\n\n\nFor OSS Maintainers: VEX Notice\n--------------------------------\nIf you\'re an OSS maintainer and Trivy has detected vulnerabilities in your project that you believe are not actually exploitable, consider issuing a VEX (Vulnerability Exploitability eXchange) statement.\nVEX allows you to communicate the actual status of vulnerabilities in your project, improving security transparency and reducing false positives for your users.\nLearn more and start using VEX: https://trivy.dev/docs/v0.70/guide/supply-chain/vex/repo#publishing-vex-documents\n\nTo disable this notice, set the TRIVY_DISABLE_VEX_NOTICE environment variable.\n\n\npo

In [7]:
agent=create_agent(model=llm,response_format=DetailedIssue,system_prompt="I'm a vulnerability analyzing and fix suggesting agent")

prompt=f"""
        analyse the jenkins output {jenkins_output_raw_text}, and give me a proper response
"""
response=agent.invoke({
    'messages':[HumanMessage(content=prompt)]
})

In [11]:
from pprint import pprint

pprint(response["structured_response"]["bom_friendly_fix"])


('Make the fix BOM-driven so transitive dependencies cannot drift.\n'
 '\n'
 'Common BOM-friendly actions:\n'
 '- **If you use Spring Boot**: upgrade `spring-boot` to a newer patch/minor '
 'version (Trivy suggests fixed versions like **3.3.11 / 3.4.5** for '
 '`org.springframework.boot:spring-boot`), which will bring compatible Spring '
 'Framework + embedded Tomcat versions.\n'
 '- Add/upgrade your BOM management section in Maven, e.g. (conceptually):\n'
 '  - Use `spring-boot-dependencies` BOM aligned to the chosen Spring Boot '
 'version.\n'
 '  - Override ONLY if needed:\n'
 '    - `log4j-core` to **2.15.0**\n'
 '    - `tomcat-embed-core` to **9.0.99** (or the line matching your Boot '
 'version)\n'
 '\n'
 'Example pattern (adjust versions to your platform support):\n'
 '- Set `dependencyManagement` to import the appropriate Spring Boot '
 'dependencies BOM.\n'
 '- Add explicit version overrides in `dependencyManagement` for `log4j-core`, '
 '`tomcat-embed-core`, and `spring-web` 